In [1]:
# ============================================================
# 15. DEMONSTRATE FARMER ASSISTANT
# ============================================================

# Step 1: Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline


# ============================================================
# Step 2: Load ACTUAL Dataset
# ============================================================

file_path = "/content/smartagri_crop_yield_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

print("\nAvailable Crops:")
print(df["Crop"].unique())


# ============================================================
# Step 3: Prepare Dataset
# ============================================================

features = [
    "Crop",
    "Soil_Type",
    "Soil_pH",
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "Temperature",
    "Rainfall",
    "Humidity"
]

target = "Yield"

X = df[features]
y = df[target]


# ============================================================
# Step 4: Encode Categorical Columns
# ============================================================

categorical_features = [
    "Crop",
    "Soil_Type"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)


# ============================================================
# Step 5: Create Yield Forecasting Model
# ============================================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)


# ============================================================
# Step 6: Train Model
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

print("\nYield forecasting model trained successfully!")


# ============================================================
# Step 7: Create Agricultural Knowledge Base
# ============================================================

knowledge_base = df.groupby("Crop").agg({
    "Soil_Type": lambda x: x.mode()[0],
    "Soil_pH": "mean",
    "Nitrogen": "mean",
    "Phosphorus": "mean",
    "Potassium": "mean",
    "Temperature": "mean",
    "Rainfall": "mean",
    "Humidity": "mean",
    "Yield": "mean"
}).reset_index()


# ============================================================
# Step 8: Farmer Assistant Function
# ============================================================

average_yield = df["Yield"].mean()


def farmer_assistant(crop):

    # Check crop
    crop_data = df[df["Crop"] == crop]

    if len(crop_data) == 0:

        print("\nCrop:", crop)
        print("Sorry, this crop is not available in the dataset.")

        return


    # --------------------------------------------------------
    # Select a REAL farm record for the selected crop
    # --------------------------------------------------------

    farm = crop_data.iloc[0]


    # --------------------------------------------------------
    # Prepare farm data for prediction
    # --------------------------------------------------------

    farm_input = pd.DataFrame([{
        "Crop": farm["Crop"],
        "Soil_Type": farm["Soil_Type"],
        "Soil_pH": farm["Soil_pH"],
        "Nitrogen": farm["Nitrogen"],
        "Phosphorus": farm["Phosphorus"],
        "Potassium": farm["Potassium"],
        "Temperature": farm["Temperature"],
        "Rainfall": farm["Rainfall"],
        "Humidity": farm["Humidity"]
    }])


    # --------------------------------------------------------
    # Predict Yield
    # --------------------------------------------------------

    predicted_yield = model.predict(farm_input)[0]


    # --------------------------------------------------------
    # Determine Yield Status
    # --------------------------------------------------------

    if predicted_yield >= average_yield * 1.10:

        yield_status = "High"

    elif predicted_yield >= average_yield * 0.90:

        yield_status = "Medium"

    else:

        yield_status = "Low"


    # --------------------------------------------------------
    # Get Knowledge Base Information
    # --------------------------------------------------------

    knowledge = knowledge_base[
        knowledge_base["Crop"] == crop
    ].iloc[0]


    # ========================================================
    # DISPLAY FARMER ASSISTANT
    # ========================================================

    print("\n")
    print("============================================================")
    print("             🌱 FARMER ADVISORY ASSISTANT")
    print("============================================================")

    print("\n--- FARM INFORMATION ---")

    print("Crop:", farm["Crop"])
    print("Soil Type:", farm["Soil_Type"])
    print("Soil pH:", round(farm["Soil_pH"], 2))
    print("Nitrogen:", round(farm["Nitrogen"], 2))
    print("Phosphorus:", round(farm["Phosphorus"], 2))
    print("Potassium:", round(farm["Potassium"], 2))
    print("Temperature:", round(farm["Temperature"], 2))
    print("Rainfall:", round(farm["Rainfall"], 2))
    print("Humidity:", round(farm["Humidity"], 2))


    # --------------------------------------------------------
    # Yield Forecast
    # --------------------------------------------------------

    print("\n--- YIELD FORECAST ---")

    print(
        "Predicted Yield:",
        round(predicted_yield, 2)
    )

    print(
        "Actual Yield:",
        round(farm["Yield"], 2)
    )

    print(
        "Yield Status:",
        yield_status
    )


    # --------------------------------------------------------
    # Agricultural Knowledge
    # --------------------------------------------------------

    print("\n--- AGRICULTURAL KNOWLEDGE ---")

    print(
        "Average Soil pH:",
        round(knowledge["Soil_pH"], 2)
    )

    print(
        "Average Nitrogen:",
        round(knowledge["Nitrogen"], 2)
    )

    print(
        "Average Phosphorus:",
        round(knowledge["Phosphorus"], 2)
    )

    print(
        "Average Potassium:",
        round(knowledge["Potassium"], 2)
    )

    print(
        "Average Rainfall:",
        round(knowledge["Rainfall"], 2)
    )

    print(
        "Average Humidity:",
        round(knowledge["Humidity"], 2)
    )


    # ========================================================
    # FARMER ADVISORY
    # ========================================================

    print("\n--- FARMER ADVISORY ---")


    # Yield advice
    if yield_status == "High":

        print(
            "✓ Yield condition is high. "
            "Continue proper crop management."
        )

    elif yield_status == "Medium":

        print(
            "✓ Yield condition is medium. "
            "Monitor soil, water and nutrients."
        )

    else:

        print(
            "✓ Yield condition is low. "
            "Improve crop management and monitor conditions."
        )


    # Soil advice
    if farm["Soil_pH"] < 6:

        print(
            "✓ Soil: pH is low. "
            "Suitable soil correction may be required."
        )

    elif farm["Soil_pH"] > 7.5:

        print(
            "✓ Soil: pH is high. "
            "Suitable soil management may be required."
        )

    else:

        print(
            "✓ Soil: pH is within the selected normal range."
        )


    # Nitrogen advice
    if farm["Nitrogen"] < 50:

        print(
            "✓ Fertilizer: Nitrogen level is low. "
            "Check nitrogen requirement."
        )

    elif farm["Nitrogen"] > 100:

        print(
            "✓ Fertilizer: Nitrogen level is high. "
            "Avoid unnecessary nitrogen application."
        )

    else:

        print(
            "✓ Fertilizer: Nitrogen condition is normal."
        )


    # Rainfall advice
    if farm["Rainfall"] < 500:

        print(
            "✓ Water: Rainfall is low. "
            "Additional irrigation may be required."
        )

    elif farm["Rainfall"] > 1200:

        print(
            "✓ Water: Rainfall is high. "
            "Monitor excess water and drainage."
        )

    else:

        print(
            "✓ Water: Rainfall condition is normal."
        )


    # Temperature advice
    if farm["Temperature"] > 35:

        print(
            "✓ Temperature: High temperature detected. "
            "Monitor heat stress and irrigation."
        )

    elif farm["Temperature"] < 20:

        print(
            "✓ Temperature: Low temperature detected. "
            "Monitor crop growth."
        )

    else:

        print(
            "✓ Temperature: No extreme temperature condition."
        )


    # Humidity advice
    if farm["Humidity"] > 80:

        print(
            "✓ Risk: High humidity may increase "
            "fungal disease risk."
        )

    elif farm["Humidity"] < 40:

        print(
            "✓ Risk: Low humidity may increase "
            "crop water requirement."
        )

    else:

        print(
            "✓ Risk: No humidity-related risk detected."
        )


    # ========================================================
    # FINAL ACTION
    # ========================================================

    print("\n--- RECOMMENDED ACTION ---")

    print(
        "Monitor soil, water, nutrients and weather "
        "conditions regularly."
    )

    print(
        "Use the predicted yield and crop information "
        "to support farm management decisions."
    )

    print("\n============================================================")
    print("Farmer assistant demonstration completed!")
    print("============================================================")


# ============================================================
# Step 9: DEMONSTRATE WITH REAL CROPS
# ============================================================

available_crops = df["Crop"].dropna().unique()

print("\nAvailable crops for demonstration:")
print(available_crops)


# Select first crop from actual dataset
selected_crop = available_crops[0]

print("\nDemonstrating Farmer Assistant for:", selected_crop)

farmer_assistant(selected_crop)


# ============================================================
# Step 10: Test Another Real Crop
# ============================================================

if len(available_crops) > 1:

    second_crop = available_crops[1]

    print("\n\nTesting another crop:", second_crop)

    farmer_assistant(second_crop)


# ============================================================
# Step 11: Unsupported Crop Test
# ============================================================

print("\n\nTesting unsupported crop:")

farmer_assistant("Banana")


# ============================================================
# Step 12: Final Project Workflow
# ============================================================

print("\n")
print("============================================================")
print("              COMPLETE PROJECT WORKFLOW")
print("============================================================")

print("""
1. Farmer / Farm Data
          ↓
2. Soil and Climate Analysis
          ↓
3. Crop Yield Forecasting
          ↓
4. Yield Status
          ↓
5. Agricultural Knowledge Base
          ↓
6. Condition-Specific Analysis
          ↓
7. LLM-Ready Advisory
          ↓
8. Farmer Recommendation
          ↓
9. Farmer Action
""")

print("============================================================")
print("       FARMER ASSISTANT DEMONSTRATION COMPLETED")
print("============================================================")

Dataset loaded successfully!
Dataset shape: (600, 10)

Available Crops:
['Millets' 'Sugarcane' 'Maize' 'Cotton' 'Rice' 'Wheat' 'Groundnut']

Yield forecasting model trained successfully!

Available crops for demonstration:
['Millets' 'Sugarcane' 'Maize' 'Cotton' 'Rice' 'Wheat' 'Groundnut']

Demonstrating Farmer Assistant for: Millets


             🌱 FARMER ADVISORY ASSISTANT

--- FARM INFORMATION ---
Crop: Millets
Soil Type: Loamy
Soil pH: 5.71
Nitrogen: 45.8
Phosphorus: 26.0
Potassium: 43.1
Temperature: 25.8
Rainfall: 402.9
Humidity: 54.0

--- YIELD FORECAST ---
Predicted Yield: 1.54
Actual Yield: 1.54
Yield Status: Low

--- AGRICULTURAL KNOWLEDGE ---
Average Soil pH: 6.13
Average Nitrogen: 41.65
Average Phosphorus: 23.46
Average Potassium: 33.63
Average Rainfall: 444.1
Average Humidity: 57.99

--- FARMER ADVISORY ---
✓ Yield condition is low. Improve crop management and monitor conditions.
✓ Soil: pH is low. Suitable soil correction may be required.
✓ Fertilizer: Nitrogen level is l